## From scratch baseline on Cypher dataset (TF-IDF + MLP)

I trained a simple neural network from scratch on the Cypher difficulty dataset, to compare against the pretrained CodeBERT model.


In [46]:
# Download datasets (google drive link)
!pip -q install gdown
import os, gdown

os.makedirs('data', exist_ok=True)
LINK_COMBINED = 'https://drive.google.com/uc?id=17yQ6zObIGYcrUlbLofqpqx1_VThE7MFO'
LINK_CURATED = 'https://drive.google.com/uc?id=1S-fIgHgUZNyCAYw_MsFjh_-y_6KiEE51'

PATH_COMBINED = 'data/combined_text2cypher.csv'
PATH_CURATED = 'data/curated_2500.csv'

gdown.download(LINK_COMBINED, PATH_COMBINED, quiet=False, fuzzy=True)

#Configuration
DATASET_CSV = PATH_COMBINED
TEXT_COL = 'cypher'
LABEL_COL = 'level'
SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

BATCH_SIZE = 64

EPOCHS = 1 #20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SUBSAMPLE = True
SUBSAMPLE_FRAC = 0.2


Downloading...
From: https://drive.google.com/uc?id=17yQ6zObIGYcrUlbLofqpqx1_VThE7MFO
To: /content/data/combined_text2cypher.csv
100%|██████████| 7.08M/7.08M [00:00<00:00, 177MB/s]


In [47]:
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [48]:
# --- Label mapping & normalization ---
LABEL_ORDER = ["nodeLevel", "1hop", "2hop", "3hop", "hardLevel"]
LABEL2ID = {name: i for i, name in enumerate(LABEL_ORDER)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}

def normalize_label(label: str) -> str:
    normalized = "".join(ch for ch in str(label).strip().lower() if ch.isalnum())
    mapping = {
        "nodelevel": "nodeLevel",
        "1hop": "1hop",
        "2hop": "2hop",
        "3hop": "3hop",
        "hard": "hardLevel",
        "hardlevel": "hardLevel",
    }
    if normalized not in mapping:
        raise ValueError(f"Unknown label '{label}'. Expected one of: {LABEL_ORDER}")
    return mapping[normalized]


In [49]:


df = pd.read_csv(PATH_COMBINED)
texts = []
labels = []
for _, r in df.iterrows():
    text = str(r[TEXT_COL]).strip()
    if not text:
        continue
    label_name = normalize_label(str(r[LABEL_COL]))
    texts.append(text)
    labels.append(LABEL2ID[label_name])

print('Loaded:', len(texts))


Loaded: 46887


In [50]:
# Stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=(1 - TRAIN_RATIO),
    random_state=SEED,
    stratify=labels,
)

val_ratio_rel = VAL_RATIO / (1 - TRAIN_RATIO)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=(1 - val_ratio_rel),
    random_state=SEED,
    stratify=y_temp,
)

print(len(X_train), len(X_val), len(X_test))


37509 4689 4689


In [51]:
df = df.sample(frac=SUBSAMPLE_FRAC, random_state=SEED)


In [52]:
df = df.groupby('level', group_keys=False).apply(
    lambda x: x.sample(frac=SUBSAMPLE_FRAC, random_state=SEED)
).reset_index(drop=True)

/tmp/ipykernel_16584/2256031241.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('level', group_keys=False).apply(


In [53]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

# Convert to torch tensors
X_train_t = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
X_val_t = torch.tensor(X_val_vec.toarray(), dtype=torch.float32)
X_test_t = torch.tensor(X_test_vec.toarray(), dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)


In [54]:
# DataLoaders
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)


In [55]:
# MLP model (from scratch)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=5, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train_t.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [56]:
# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for xb, yb in train_loader:
        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1).detach().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    train_f1 = f1_score(all_labels, all_preds, average='macro')

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            preds = torch.argmax(logits, dim=-1).detach().numpy()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"Epoch {epoch+1}/{EPOCHS} | loss {total_loss/len(train_loader):.4f} | train acc {train_acc:.4f} f1 {train_f1:.4f} | val acc {val_acc:.4f} f1 {val_f1:.4f}")


Epoch 1/1 | loss 0.4051 | train acc 0.8509 f1 0.7433 | val acc 0.9467 f1 0.8962


In [57]:
# Test evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = torch.argmax(logits, dim=-1).detach().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Test acc {test_acc:.4f} f1 {test_f1:.4f}")


Test acc 0.9446 f1 0.9003


In [58]:
# 1) Load curated CSV
df_cur = pd.read_csv(PATH_CURATED)

# 2) Keep only hardLevel
texts = []
labels = []
for _, row in df_cur.iterrows():
    label_name = normalize_label(row.get('level',''))
    if label_name != 'hardLevel':
        continue
    text = str(row.get('cypher','')).strip()
    if not text:
        continue
    texts.append(text)
    labels.append(LABEL2ID['hardLevel'])

# 3) TF‑IDF transform (same vectorizer used in training)
X = vectorizer.transform(texts)
X = torch.tensor(X.toarray(), dtype=torch.float32)

# 4) Predict
model.eval()
with torch.no_grad():
    logits = model(X)
    preds = torch.argmax(logits, dim=-1).cpu().numpy()

# 5) Metrics for hardLevel
acc = accuracy_score(labels, preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    labels, preds, labels=[LABEL2ID['hardLevel']], average=None
)

print("HardLevel acc:", acc)
print(f"HardLevel precision: {prec[0]:.4f}")
print(f"HardLevel recall:    {rec[0]:.4f}")
print(f"HardLevel F1:        {f1[0]:.4f}")


HardLevel acc: 0.948
HardLevel precision: 1.0000
HardLevel recall:    0.9480
HardLevel F1:        0.9733


In [59]:
query =   "MATCH (m:pre_miRNA)-[:is_causal_somatic_mutation_in]->(p) WHERE (p:Phenotype OR (p)-[:subclassof*1..5]->(:Phenotype)) AND p.Label CONTAINS 'Thyroid' RETURN DISTINCT m.Label AS pre_miRNA, p"

# Vectorize with the SAME trained vectorizer
x = vectorizer.transform([query])
x = torch.tensor(x.toarray(), dtype=torch.float32)

model.eval()
with torch.no_grad():
    logits = model(x)
    pred = torch.argmax(logits, dim=-1).item()

print("Predicted level:", ID2LABEL[pred])


Predicted level: hardLevel


the predicted 'hardLevel' is correct